In [1]:
from heapq import heappush, heappop

# Initial and Goal states
initial = (
    (2, 8, 3),
    (1, 6, 4),
    (7, 0, 5)   # 0 = blank
)

goal = (
    (1, 2, 3),
    (8, 0, 4),
    (7, 6, 5)
)

moves = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1)
}

opposite = {
    "UP": "DOWN",
    "DOWN": "UP",
    "LEFT": "RIGHT",
    "RIGHT": "LEFT"
}


# Generate possible next states
def get_neighbors(state, previous_move=None):
    r, c = next(
        (r, c)
        for r in range(3)
        for c in range(3)
        if state[r][c] == 0
    )

    neighbors = []

    for move, (dr, dc) in moves.items():

        # Do not immediately reverse the previous move
        if previous_move and move == opposite[previous_move]:
            continue

        nr, nc = r + dr, c + dc

        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = [list(row) for row in state]

            new_state[r][c], new_state[nr][nc] = (
                new_state[nr][nc],
                new_state[r][c]
            )

            neighbors.append((move, tuple(map(tuple, new_state))))

    return neighbors


# Case 1: Number of misplaced tiles
def misplaced_tiles(state):
    count = 0

    for r in range(3):
        for c in range(3):
            if state[r][c] != 0 and state[r][c] != goal[r][c]:
                count += 1

    return count


# Case 2: Manhattan Distance
def manhattan_distance(state):
    distance = 0

    for r in range(3):
        for c in range(3):
            value = state[r][c]

            if value != 0:
                gr, gc = next(
                    (x, y)
                    for x in range(3)
                    for y in range(3)
                    if goal[x][y] == value
                )

                distance += abs(r - gr) + abs(c - gc)

    return distance


# Print board
def print_board(state, indent=""):
    for row in state:
        print(indent + " ".join("_" if x == 0 else str(x) for x in row))


# Print complete state-space tree up to depth 5
def print_tree(state, depth=0, previous_move=None, path=None):
    if path is None:
        path = []

    indent = "    " * depth

    h1 = misplaced_tiles(state)
    h2 = manhattan_distance(state)

    f1 = depth + h1
    f2 = depth + h2

    print(indent + "├── State")
    print(indent + f"│   g = {depth}, h1 = {h1}, f1 = {f1}")
    print(indent + f"│   g = {depth}, h2 = {h2}, f2 = {f2}")

    print_board(state, indent + "│   ")

    if state == goal:
        print(indent + "│   ★ GOAL")
        print(indent + "│   Path:", " → ".join(path))
        return

    if depth == 5:
        return

    for move, next_state in get_neighbors(state, previous_move):

        print(indent + f"│")
        print(indent + f"└── {move}")

        print_tree(
            next_state,
            depth + 1,
            move,
            path + [move]
        )


# -----------------------------
# A* SEARCH
# -----------------------------

def a_star(heuristic):
    open_list = []

    counter = 0

    h = heuristic(initial)

    heappush(
        open_list,
        (h, counter, 0, initial, [], None)
    )

    visited = set()

    while open_list:

        f, _, g, state, path, previous_move = heappop(open_list)

        if state in visited:
            continue

        visited.add(state)

        if state == goal:
            return path, g

        for move, next_state in get_neighbors(state, previous_move):

            if next_state not in visited:

                new_g = g + 1
                new_h = heuristic(next_state)
                new_f = new_g + new_h

                counter += 1

                heappush(
                    open_list,
                    (
                        new_f,
                        counter,
                        new_g,
                        next_state,
                        path + [move],
                        move
                    )
                )

    return None, None


# -----------------------------
# MAIN
# -----------------------------

print("=" * 70)
print("A* SEARCH - 8 PUZZLE")
print("=" * 70)

print("\nINITIAL STATE:")
print_board(initial)

print("\nGOAL STATE:")
print_board(goal)

print("\n" + "=" * 70)
print("CASE 1: NUMBER OF MISPLACED TILES")
print("=" * 70)

path1, cost1 = a_star(misplaced_tiles)

print("\nA* Solution:")
print(" → ".join(path1))
print("Cost:", cost1)

print("\nFULL STATE-SPACE TREE:")
print_tree(initial)


print("\n\n" + "=" * 70)
print("CASE 2: MANHATTAN DISTANCE")
print("=" * 70)

path2, cost2 = a_star(manhattan_distance)

print("\nA* Solution:")
print(" → ".join(path2))
print("Cost:", cost2)

print("\nFULL STATE-SPACE TREE:")
print_tree(initial)

A* SEARCH - 8 PUZZLE

INITIAL STATE:
2 8 3
1 6 4
7 _ 5

GOAL STATE:
1 2 3
8 _ 4
7 6 5

CASE 1: NUMBER OF MISPLACED TILES

A* Solution:
UP → UP → LEFT → DOWN → RIGHT
Cost: 5

FULL STATE-SPACE TREE:
├── State
│   g = 0, h1 = 4, f1 = 4
│   g = 0, h2 = 5, f2 = 5
│   2 8 3
│   1 6 4
│   7 _ 5
│
└── UP
    ├── State
    │   g = 1, h1 = 3, f1 = 4
    │   g = 1, h2 = 4, f2 = 5
    │   2 8 3
    │   1 _ 4
    │   7 6 5
    │
    └── UP
        ├── State
        │   g = 2, h1 = 3, f1 = 5
        │   g = 2, h2 = 3, f2 = 5
        │   2 _ 3
        │   1 8 4
        │   7 6 5
        │
        └── LEFT
            ├── State
            │   g = 3, h1 = 2, f1 = 5
            │   g = 3, h2 = 2, f2 = 5
            │   _ 2 3
            │   1 8 4
            │   7 6 5
            │
            └── DOWN
                ├── State
                │   g = 4, h1 = 1, f1 = 5
                │   g = 4, h2 = 1, f2 = 5
                │   1 2 3
                │   _ 8 4
                │   7 6 5
               